In [1]:
from robustbench.utils import load_model
import torch.nn as nn
import time
import torch.nn as nn
import torchvision
import torch.nn.functional as F
from torchvision import datasets, transforms
import torch
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torch.optim as optim

In [2]:
from robustbench.utils import load_model
model = load_model(model_name='Standard', model_dir = "./ckpt", dataset='cifar10', threat_model='corruptions')
# 修改模型的第一层卷积核
# model.conv1 = nn.Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)

def initialize_weights(model):
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

initialize_weights(model)
model = model.cuda()
# save model
# torch.save(model.state_dict(), "ckpt/svhn/test.pth")
# load model
model.load_state_dict(torch.load("ckpt/svhn/my_train_10_3_best.pth"))

ckpt/cifar10/corruptions/Standard.pt
dict_keys(['epoch', 'arch', 'state_dict', 'best_prec1', 'optimizer'])


<All keys matched successfully>

In [2]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import time

# 数据加载
# transform_chain = transforms.Compose( 
#         [transforms.ToTensor(),
#         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
transform_chain = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_chain)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# 优化器
# optimizer = optim.Adam(model.parameters(), lr=0.0001)
optimizer = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)

# 训练循环
model.train()
for epoch in range(10):  # 训练10个epoch
    start_time = time.time()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        data, target = data.cuda(), target.cuda()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            elapsed_time = time.time() - start_time
            remaining_batches = len(train_loader) - batch_idx - 1
            estimated_time = elapsed_time / (batch_idx + 1) * remaining_batches
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}\tElapsed Time: {elapsed_time:.2f}s\tEstimated Time Remaining: {estimated_time:.2f}s')


Files already downloaded and verified
Train Epoch: 0 [0/50000 (0%)]	Loss: 0.001313	Elapsed Time: 0.51s	Estimated Time Remaining: 400.51s
Train Epoch: 0 [6400/50000 (13%)]	Loss: 0.000469	Elapsed Time: 33.42s	Estimated Time Remaining: 225.35s
Train Epoch: 0 [12800/50000 (26%)]	Loss: 0.000385	Elapsed Time: 67.01s	Estimated Time Remaining: 193.71s
Train Epoch: 0 [19200/50000 (38%)]	Loss: 0.001927	Elapsed Time: 100.41s	Estimated Time Remaining: 160.45s
Train Epoch: 0 [25600/50000 (51%)]	Loss: 0.000610	Elapsed Time: 133.58s	Estimated Time Remaining: 126.92s
Train Epoch: 0 [32000/50000 (64%)]	Loss: 0.000901	Elapsed Time: 167.13s	Estimated Time Remaining: 93.74s
Train Epoch: 0 [38400/50000 (77%)]	Loss: 0.000997	Elapsed Time: 200.71s	Estimated Time Remaining: 60.45s
Train Epoch: 0 [44800/50000 (90%)]	Loss: 0.002870	Elapsed Time: 233.77s	Estimated Time Remaining: 27.01s
Train Epoch: 1 [0/50000 (0%)]	Loss: 0.000214	Elapsed Time: 0.06s	Estimated Time Remaining: 49.54s


KeyboardInterrupt: 

In [3]:
# save model
torch.save(model.state_dict(), "ckpt/cifar10/corruptions/my_train.pth")
# load model
# model.load_state_dict(torch.load("model.pth"))

: 

In [3]:
# 定义图像转换
transform_svhn = transforms.Compose([
    transforms.Resize((32, 32)),  # 调整图像大小
    # transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),  # 转换为张量
    # transforms.Normalize(mean=[0.4377, 0.4438, 0.4728], std=[0.1980, 0.2010, 0.1970])
    # transforms.Normalize(mean=[0.4453], std=[0.1970])
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 归一化
])

transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),  # 调整图像大小
    transforms.Grayscale(num_output_channels=3),  # 将单通道转换为三通道
    transforms.ToTensor(),  # 转换为张量
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 归一化
])

# 加载SVHN数据集
svhn_train = datasets.SVHN(root='./data', split='train', transform=transform_svhn, download=True)
svhn_test = datasets.SVHN(root='./data', split='test', transform=transform_svhn, download=True)

# 加载MNIST数据集
mnist_train = datasets.MNIST(root='./data', train=True, transform=transform_mnist, download=True)
mnist_test = datasets.MNIST(root='./data', train=False, transform=transform_mnist, download=True)

# 创建数据加载器
trainloader = torch.utils.data.DataLoader(dataset=svhn_train, batch_size=128, shuffle=True, num_workers=4)
testloader = torch.utils.data.DataLoader(dataset=mnist_test, batch_size=128, shuffle=False, num_workers=4)


# 定义ResNet模型
# model = resnet18(num_classes=10)
# model = model.cuda()

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)

# 学习率调度器
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)
# sche
# 训练函数
def train(epoch):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.cuda(), targets.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch} | Batch: {batch_idx} | Loss: {train_loss/(batch_idx+1):.3f} | Acc: {100.*correct/total:.3f}%')

# 测试函数
best_acc = 0
def test(epoch):
    model.eval()
    global best_acc
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    print(f'Test Epoch: {epoch} | Loss: {test_loss/(batch_idx+1):.3f} | Acc: {100.*correct/total:.3f}%')
    if 100.*correct/total > best_acc:
        # torch.save(model.state_dict(), "ckpt/svhn/my_train_10_.pth")
        torch.save(model.state_dict(), "ckpt/svhn/my_train_10_3_best.pth")
        print("Model saved!")
        best_acc = 100.*correct/total

# 训练和测试循环
for epoch in range(60):
    train(epoch)
    test(epoch)
    scheduler.step()

torch.save(model.state_dict(), "ckpt/svhn/my_train_10_3_final.pth")

Using downloaded and verified file: ./data/train_32x32.mat
Using downloaded and verified file: ./data/test_32x32.mat


/home/q/miniconda3/envs/tta/lib/python3.9/site-packages/torchvision/datasets/mnist.py:498: UserWarning: The given NumPy array is not writeable, and PyTorch does not support non-writeable tensors. This means you can write to the underlying (supposedly non-writeable) NumPy array using the tensor. You may want to copy the array to protect its data or make it writeable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at  /pytorch/torch/csrc/utils/tensor_numpy.cpp:180.)
  return torch.from_numpy(parsed.astype(m[2], copy=False)).view(*s)


Epoch: 0 | Batch: 0 | Loss: 9.295 | Acc: 9.375%
Epoch: 0 | Batch: 100 | Loss: 3.045 | Acc: 19.562%
Epoch: 0 | Batch: 200 | Loss: 2.496 | Acc: 26.737%
Epoch: 0 | Batch: 300 | Loss: 2.176 | Acc: 34.484%
Epoch: 0 | Batch: 400 | Loss: 1.917 | Acc: 41.887%
Epoch: 0 | Batch: 500 | Loss: 1.715 | Acc: 47.896%
Test Epoch: 0 | Loss: 2.452 | Acc: 27.840%
Model saved!
Epoch: 1 | Batch: 0 | Loss: 0.576 | Acc: 82.812%
Epoch: 1 | Batch: 100 | Loss: 0.627 | Acc: 80.616%
Epoch: 1 | Batch: 200 | Loss: 0.589 | Acc: 81.985%
Epoch: 1 | Batch: 300 | Loss: 0.565 | Acc: 82.675%
Epoch: 1 | Batch: 400 | Loss: 0.544 | Acc: 83.368%
Epoch: 1 | Batch: 500 | Loss: 0.522 | Acc: 84.054%
Test Epoch: 1 | Loss: 3.024 | Acc: 38.280%
Model saved!
Epoch: 2 | Batch: 0 | Loss: 0.326 | Acc: 93.750%
Epoch: 2 | Batch: 100 | Loss: 0.360 | Acc: 89.109%
Epoch: 2 | Batch: 200 | Loss: 0.351 | Acc: 89.408%
Epoch: 2 | Batch: 300 | Loss: 0.348 | Acc: 89.491%
Epoch: 2 | Batch: 400 | Loss: 0.344 | Acc: 89.590%
Epoch: 2 | Batch: 500 | Loss

KeyboardInterrupt: 

In [4]:
torch.save(model.state_dict(), "ckpt/svhn/my_train_10_3_final.pth")

In [9]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为张量
    # resize the image to 32x32
    transforms.Resize((32, 32)),
    # pad to 32 x 32
    # transforms.Pad(2, fill=0, padding_mode='constant'),
    # To3ChannelsWithPad(padding=padding)  # 转换为三通道并填充
])

# 加载MNIST测试集
mnist_testset = MNIST(root='./data', train=False, download=True, transform=mnist_transform)
mnist_testloader = DataLoader(mnist_testset, batch_size=100, shuffle=False, num_workers=2)

# 在MNIST数据集上测试
model.eval()
mnist_test_loss = 0
mnist_correct = 0
mnist_total = 0
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(mnist_testloader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # print(inputs.shape)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        mnist_test_loss += loss.item()
        _, predicted = outputs.max(1)
        mnist_total += targets.size(0)
        mnist_correct += predicted.eq(targets).sum().item()

print(f'MNIST Test Loss: {mnist_test_loss/(batch_idx+1):.3f} | Acc: {100.*mnist_correct/mnist_total:.3f}%')

MNIST Test Loss: 1.746 | Acc: 54.910%


In [4]:
from cifar10c import setup_source, setup_norm, setup_tent

In [6]:
import copy
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # 调整图像大小
    transforms.Grayscale(num_output_channels=3),  # 将单通道转换为三通道
    transforms.ToTensor(),  # 转换为张量
    # transforms.Normalize(mean=[0.4453], std=[0.1970]),
    # transforms.Normalize(mean=[0.4377, 0.4438, 0.4728], std=[0.1980, 0.2010, 0.1970])
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 归一化
    # transforms.Normalize((0.5), (0.5))  # 归一化
])

# 加载MNIST测试集
mnist_testset = MNIST(root='./data', train=False, download=True, transform=mnist_transform)
# mnist_testset = datasets.SVHN(root='./data', split='test', transform=mnist_transform, download=True)
mnist_testloader = DataLoader(mnist_testset, batch_size=128, shuffle=False, num_workers=2)

# 在MNIST数据集上测试

b_model = copy.deepcopy(model)
# b_model = setup_source(b_model)
b_model = setup_norm(b_model)
# b_model = setup_tent(b_model)
criterion = nn.CrossEntropyLoss()
mnist_test_loss = 0
mnist_correct = 0
mnist_total = 0
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(mnist_testloader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # print(inputs.shape)
        outputs = b_model(inputs)
        loss = criterion(outputs, targets)

        mnist_test_loss += loss.item()
        _, predicted = outputs.max(1)
        mnist_total += targets.size(0)
        mnist_correct += predicted.eq(targets).sum().item()

print(f'MNIST Test Loss: {mnist_test_loss/(batch_idx+1):.3f} | Acc: {100.*mnist_correct/mnist_total:.3f}%')

MNIST Test Loss: 1.575 | Acc: 58.130%


In [9]:
import copy
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # 调整图像大小
    transforms.Grayscale(num_output_channels=3),  # 将单通道转换为三通道
    transforms.ToTensor(),  # 转换为张量
    # transforms.Normalize(mean=[0.4453], std=[0.1970]),
    # transforms.Normalize(mean=[0.4377, 0.4438, 0.4728], std=[0.1980, 0.2010, 0.1970])
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 归一化
    # transforms.Normalize((0.5), (0.5))  # 归一化
])

# 加载MNIST测试集
mnist_testset = MNIST(root='./data', train=False, download=True, transform=mnist_transform)
# mnist_testset = datasets.SVHN(root='./data', split='test', transform=mnist_transform, download=True)
mnist_testloader = DataLoader(mnist_testset, batch_size=128, shuffle=False, num_workers=2)

# 在MNIST数据集上测试

b_model = copy.deepcopy(model)
b_model = setup_source(b_model)
# b_model = setup_norm(b_model)
# b_model = setup_tent(b_model)
criterion = nn.CrossEntropyLoss()
mnist_test_loss = 0
mnist_correct = 0
mnist_total = 0
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(mnist_testloader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # print(inputs.shape)
        outputs = b_model(inputs)
        loss = criterion(outputs, targets)

        mnist_test_loss += loss.item()
        _, predicted = outputs.max(1)
        mnist_total += targets.size(0)
        mnist_correct += predicted.eq(targets).sum().item()

print(f'MNIST Test Loss: {mnist_test_loss/(batch_idx+1):.3f} | Acc: {100.*mnist_correct/mnist_total:.3f}%')

MNIST Test Loss: 1.512 | Acc: 65.730%


In [19]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# MNIST 和 SVHN 的均值和标准差
mnist_mean = [0.1327, 0.1327, 0.1327]
mnist_std = [0.2919, 0.2919, 0.2919]
svhn_mean = [0.4377, 0.4438, 0.4728]
svhn_std = [0.1980, 0.2010, 0.1970]

# 定义归一化和反归一化变换
normalize_mnist = transforms.Normalize(mean=mnist_mean, std=mnist_std)
denormalize_to_svhn = transforms.Lambda(
    lambda img: transforms.functional.normalize(img, [-m/s for m, s in zip(svhn_mean, svhn_std)], [1/s for s in svhn_std])
    # lambda img: transforms.functional.normalize(img, [-m/s for m, s in zip(svhn_mean, svhn_std)], svhn_std)
)

# 定义转换函数
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # 调整图像大小
    transforms.Grayscale(num_output_channels=3),  # 将单通道转换为三通道
    transforms.ToTensor(),  # 转换为张量
    # normalize_mnist,  # 使用MNIST的均值和标准差进行归一化
    # denormalize_to_svhn  # 使用SVHN的均值和标准差进行反归一化
])

# 加载MNIST测试集并转换为三通道
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)

# 创建数据加载器
mnist_testloader = DataLoader(mnist_testset, batch_size=128, shuffle=False, num_workers=2)

b_model = copy.deepcopy(model)
# b_model = setup_source(b_model)
# b_model = setup_norm(b_model)
b_model = setup_tent(b_model)
criterion = nn.CrossEntropyLoss()
mnist_test_loss = 0
mnist_correct = 0
mnist_total = 0
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(mnist_testloader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # print(inputs.shape)
        outputs = b_model(inputs)
        loss = criterion(outputs, targets)

        mnist_test_loss += loss.item()
        _, predicted = outputs.max(1)
        mnist_total += targets.size(0)
        mnist_correct += predicted.eq(targets).sum().item()

print(f'MNIST Test Loss: {mnist_test_loss/(batch_idx+1):.3f} | Acc: {100.*mnist_correct/mnist_total:.3f}%')

1
False
MNIST Test Loss: 2.595 | Acc: 53.870%


In [7]:
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # 调整图像大小
    transforms.Grayscale(num_output_channels=3),  # 将单通道转换为三通道
    transforms.ToTensor(),  # 转换为张量
    # transforms.Normalize(mean=[0.4453], std=[0.1970]),
    # transforms.Normalize(mean=[0.4377, 0.4438, 0.4728], std=[0.1980, 0.2010, 0.1970])
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 归一化
    # transforms.Normalize((0.5), (0.5))  # 归一化
    # transforms.Normalize([-0.7345], [0.5838])  # 归一化
])
mnist_testset = MNIST(root='./data', train=False, download=True, transform=mnist_transform)
# mnist_testset = datasets.SVHN(root='./data', split='test', transform=mnist_transform, download=True)
mnist_testloader = DataLoader(mnist_testset, batch_size=128, shuffle=False, num_workers=2)
# 将数据转换为Tensor，这里可以迭代dataset来获取灰度图像
svhn_train_data = torch.stack([img for img, _ in mnist_testset], dim=0)
# svhn_test_data = torch.stack([img for img, _ in mnist_testloader], dim=0)

# 合并训练和测试数据集
svhn_data = svhn_train_data

# 计算每个通道的均值和标准差
print(svhn_data.shape)
mean = svhn_data.mean(dim=[0, 2, 3])
std = svhn_data.std(dim=[0, 2, 3])

print(f'Mean: {mean}')
print(f'Std: {std}')

torch.Size([10000, 3, 32, 32])
Mean: tensor([0.1327, 0.1327, 0.1327])
Std: tensor([0.2919, 0.2919, 0.2919])
